# Adaptive Top-K via Reranker Score Gap — Analysis

**Goal:** Find a data-driven cutoff strategy to replace the fixed `top_k_final=10`.

**Problem:** Val median citations = 22, max = 47. Fixed top-10 structurally caps recall.  
**Hypothesis:** After reranking, relevant docs cluster at the top and there is a detectable score "cliff" separating them from noise. We can cut at that cliff instead of a fixed rank.

**Plan:**
1. Measure retriever recall@K (does top-80 even contain all gold citations?)
2. Run full pipeline on training set — retrieve top-80, rerank, store full scored lists
3. Per-query: visualise score distributions, where gold citations land
4. Aggregate: does the largest score gap correlate with the true cutoff?
5. Compare gap strategies (largest gap / relative threshold / absolute threshold) by F1
6. Apply best strategy to val, compare against fixed top-10

In [ ]:
!pip install -q sentence_transformers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import gc
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import MarianMTModel, MarianTokenizer

warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')
plt.rcParams['figure.dpi'] = 110

In [ ]:
BASE = '/kaggle/input/llm-agentic-legal-information-retrieval'

val_path    = f'{BASE}/val.csv'
test_path   = f'{BASE}/test.csv'
laws_path   = f'{BASE}/laws_de.csv'
output_path = 'submission.csv'

In [ ]:
embedding_model_name   = 'BAAI/bge-m3'
reranker_model_name    = 'BAAI/bge-reranker-v2-m3'
translation_model_name = 'Helsinki-NLP/opus-mt-en-de'

use_gpu  = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if use_gpu else 0

if num_gpus >= 2:
    device_emb, device_rerank, device_transl = 'cuda:0', 'cuda:1', 'cuda:1'
elif num_gpus == 1:
    device_emb = device_rerank = device_transl = 'cuda:0'
else:
    device_emb = device_rerank = device_transl = 'cpu'

print(f'devices — emb: {device_emb}, rerank: {device_rerank}, transl: {device_transl}')

TOP_K_RETRIEVAL  = 80     # candidates fetched by embedder (val max citations = 47, +33 buffer)
TEXT_TRUNCATE    = 2000
TRANSL_MAX_CHARS = 1500
RERANK_BATCH     = 16

In [ ]:
BASE = '/kaggle/input/llm-agentic-legal-information-retrieval'

train_df = pd.read_csv(f'{BASE}/train.csv')
val_df   = pd.read_csv(f'{BASE}/val.csv')
laws_df  = pd.read_csv(f'{BASE}/laws_de.csv')
laws_df['text'] = laws_df['text'].fillna('')

corpus_cits = set(laws_df['citation'])

# Citation count per query
train_df['n_citations'] = train_df['gold_citations'].str.split(';').str.len()
val_df['n_citations']   = val_df['gold_citations'].str.split(';').str.len()

# Filter train to queries where ALL gold citations exist in the corpus
# (28.8% of train labels are missing — exclude those to avoid false negatives)
def all_in_corpus(cit_str):
    return all(c.strip() in corpus_cits for c in cit_str.split(';'))

train_clean = train_df[train_df['gold_citations'].apply(all_in_corpus)].reset_index(drop=True)

print(f'laws:        {len(laws_df):,} articles')
print(f'train total: {len(train_df):,}  |  train clean (all cits in corpus): {len(train_clean):,}')
print(f'val:         {len(val_df):,}')
print(f'\ntrain_clean citation count — median: {train_clean["n_citations"].median():.0f}, '
      f'max: {train_clean["n_citations"].max()}')
print(f'val citation count         — median: {val_df["n_citations"].median():.0f}, '
      f'max: {val_df["n_citations"].max()}')

In [ ]:
def load_translator():
    print(f'loading translator on {device_transl}')
    tokenizer = MarianTokenizer.from_pretrained(translation_model_name)
    model     = MarianMTModel.from_pretrained(translation_model_name).to(device_transl)
    model.eval()
    print('translator ready')
    return tokenizer, model


def translate_to_german(text: str, tokenizer, model) -> str:
    """Translate English text to German. Chunks long inputs to stay within model limits."""
    # Split into chunks at paragraph/sentence boundaries
    chunks, current = [], []
    current_len = 0
    for para in text.split('\n'):
        if current_len + len(para) > transl_max_chars and current:
            chunks.append('\n'.join(current))
            current, current_len = [], 0
        current.append(para)
        current_len += len(para)
    if current:
        chunks.append('\n'.join(current))

    translated_chunks = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        inputs = tokenizer(
            [chunk], return_tensors='pt',
            padding=True, truncation=True, max_length=512
        ).to(device_transl)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=512)
        translated_chunks.append(tokenizer.decode(output_ids[0], skip_special_tokens=True))

    return '\n'.join(translated_chunks)

In [ ]:
def load_embedding_model():
    print(f'loading embedder on {device_emb}')
    model = SentenceTransformer(
        embedding_model_name,
        device=device_emb,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print('embedder ready')
    return model


def encode_corpus(model, texts):
    truncated = [str(t)[:text_truncate] for t in texts]
    return model.encode(
        truncated,
        batch_size=laws_batch_size,
        convert_to_tensor=False,
        normalize_embeddings=True,
        show_progress_bar=True
    )


def encode_query(model, query):
    return model.encode(
        query,
        convert_to_tensor=True,
        normalize_embeddings=True
    ).cpu().numpy()


def search_dense(query_embedding, corpus_embeddings, top_k):
    scores      = np.dot(corpus_embeddings, query_embedding)
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [(int(idx), float(scores[idx])) for idx in top_indices]

In [ ]:
def load_reranker():
    print(f'loading reranker on {device_rerank}')
    model = CrossEncoder(
        reranker_model_name,
        device=device_rerank,
        max_length=512,
        model_kwargs={'torch_dtype': torch.float16}
    )
    print('reranker ready')
    return model


def rerank_with_scores(reranker, query: str, candidates: list[tuple]) -> list[tuple]:
    """
    candidates: list of (citation, text)
    returns: list of (citation, score) sorted descending by score
    """
    if not candidates:
        return []
    pairs     = [[query, str(text)[:TEXT_TRUNCATE]] for _, text in candidates]
    citations = [cit for cit, _ in candidates]
    scores    = reranker.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
    ranked    = sorted(zip(citations, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked  # [(citation, score), ...]

In [ ]:
def encode_corpus(model, texts):
    truncated = [str(t)[:TEXT_TRUNCATE] for t in texts]
    return model.encode(
        truncated, batch_size=8, convert_to_tensor=False,
        normalize_embeddings=True, show_progress_bar=True
    )

def encode_query(model, query):
    return model.encode(query, convert_to_tensor=True, normalize_embeddings=True).cpu().numpy()

def retrieve_top_k(query_emb, corpus_embs, k):
    scores  = np.dot(corpus_embs, query_emb)
    top_idx = np.argsort(scores)[-k:][::-1]
    return [(int(i), float(scores[i])) for i in top_idx]

def f1_score(predicted: set, gold: set) -> float:
    if not gold:
        return 0.0
    tp = len(predicted & gold)
    if tp == 0:
        return 0.0
    p = tp / len(predicted)
    r = tp / len(gold)
    return 2 * p * r / (p + r)

def apply_cutoff(ranked_scores: list[tuple], strategy: str, param=None) -> list[str]:
    """
    ranked_scores: [(citation, score), ...] sorted descending
    strategy: 'fixed_k' | 'largest_gap' | 'gap_threshold' | 'relative'
    param: k for fixed_k, delta for gap_threshold, alpha for relative
    returns: list of selected citations
    """
    if not ranked_scores:
        return []
    scores = [s for _, s in ranked_scores]
    cits   = [c for c, _ in ranked_scores]

    if strategy == 'fixed_k':
        return cits[:param]

    if strategy == 'largest_gap':
        gaps = [scores[i] - scores[i+1] for i in range(len(scores)-1)]
        cut  = int(np.argmax(gaps)) + 1  # cut after this rank (1-indexed)
        return cits[:cut]

    if strategy == 'gap_threshold':
        # take all docs until gap > param appears
        cut = len(scores)
        for i in range(len(scores)-1):
            if scores[i] - scores[i+1] > param:
                cut = i + 1
                break
        return cits[:cut]

    if strategy == 'relative':
        # take all docs with score >= alpha * top_score
        threshold = param * scores[0]
        return [c for c, s in ranked_scores if s >= threshold]

    raise ValueError(f'unknown strategy: {strategy}')

In [ ]:
# ── Phase 0: encode corpus (reused across all sections) ─────────────────────
emb_model = load_embedding_model()
print('encoding laws corpus...')
corpus_embs = encode_corpus(emb_model, laws_df['text'].tolist())
print(f'corpus embeddings shape: {corpus_embs.shape}')

## Section 1 — Retriever Recall@K

Before reranking: what fraction of gold citations appear in the top-K retrieved by the embedder?  
This sets the hard ceiling — the reranker cannot recover citations that aren't in the candidate pool.

In [ ]:
Ks = [10, 20, 30, 50, 60, 80, 100]

# Use a sample of clean train queries for speed (or all if small enough)
sample = train_clean.sample(min(200, len(train_clean)), random_state=42)

recall_at_k = {k: [] for k in Ks}

for _, row in sample.iterrows():
    gold = set(c.strip() for c in row['gold_citations'].split(';'))
    q_emb = encode_query(emb_model, str(row['query'])[:TEXT_TRUNCATE])
    # retrieve top-100 once, then slice
    top100 = retrieve_top_k(q_emb, corpus_embs, 100)
    retrieved_cits_100 = [laws_df.iloc[idx]['citation'] for idx, _ in top100]
    for k in Ks:
        retrieved_k = set(retrieved_cits_100[:k])
        recall_at_k[k].append(len(gold & retrieved_k) / len(gold))

fig, ax = plt.subplots(figsize=(10, 5))
means = [np.mean(recall_at_k[k]) for k in Ks]
ax.plot(Ks, means, marker='o', linewidth=2, color='steelblue')
for k, m in zip(Ks, means):
    ax.annotate(f'{m:.2%}', (k, m), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
ax.axvline(80, color='tomato', linestyle='--', label='TOP_K_RETRIEVAL=80')
ax.set_xlabel('K (candidates retrieved by embedder)')
ax.set_ylabel('Mean Recall@K (fraction of gold citations found)')
ax.set_title('Retriever recall@K — train_clean sample')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('\nMean retriever recall@K:')
for k, m in zip(Ks, means):
    print(f'  @{k:3d}: {m:.3f}')

## Section 2 — Run full pipeline on train_clean, collect ranked score lists

Training queries are already German — pass directly to reranker (no translation needed).  
Store the full `(citation, score, is_gold)` list for every query.

In [ ]:
reranker = load_reranker()

train_results = []  # list of dicts, one per query

for i, row in train_clean.iterrows():
    gold = set(c.strip() for c in row['gold_citations'].split(';'))
    query = str(row['query'])[:TEXT_TRUNCATE]  # already German

    # Retrieve top-80
    q_emb    = encode_query(emb_model, query)
    top_hits = retrieve_top_k(q_emb, corpus_embs, TOP_K_RETRIEVAL)
    candidates = [(laws_df.iloc[idx]['citation'], laws_df.iloc[idx]['text']) for idx, _ in top_hits]

    # Rerank — get full scored list
    ranked = rerank_with_scores(reranker, query, candidates)  # [(cit, score), ...]

    # Annotate with is_gold
    annotated = [(cit, score, cit in gold) for cit, score in ranked]

    # Retriever recall@80 (before reranking)
    retrieved_cits = set(c for c, _ in candidates)
    retriever_recall = len(gold & retrieved_cits) / len(gold)

    # Optimal cutoff = rank that maximises F1
    best_f1, best_cut = 0.0, 1
    for cut in range(1, len(annotated) + 1):
        pred = set(c for c, _, _ in annotated[:cut])
        f = f1_score(pred, gold)
        if f >= best_f1:
            best_f1 = f
            best_cut = cut

    # Largest gap position
    scores_list = [s for _, s, _ in annotated]
    gaps        = [scores_list[j] - scores_list[j+1] for j in range(len(scores_list)-1)]
    lg_pos      = int(np.argmax(gaps)) + 1 if gaps else 1

    train_results.append({
        'query_id':         row['query_id'],
        'n_gold':           len(gold),
        'retriever_recall': retriever_recall,
        'ranked':           annotated,           # [(cit, score, is_gold), ...]
        'optimal_cut':      best_cut,
        'optimal_f1':       best_f1,
        'largest_gap_pos':  lg_pos,
        'last_gold_rank':   max((j+1 for j, (_, _, g) in enumerate(annotated) if g), default=0),
    })

    if (len(train_results)) % 50 == 0:
        print(f'  {len(train_results)}/{len(train_clean)} done')

print(f'\nDone — {len(train_results)} queries processed')

## Section 3 — Per-query score visualisation (sample of 12)

For a representative sample: score by rank, gold citations highlighted in green.

In [ ]:
# Sample 12 queries spread across citation-count buckets
buckets = [(1,1), (2,3), (4,8), (9,44)]
sample_results = []
for lo, hi in buckets:
    subset = [r for r in train_results if lo <= r['n_gold'] <= hi]
    sample_results.extend(subset[:3])

fig, axes = plt.subplots(3, 4, figsize=(22, 12))

for ax, res in zip(axes.flat, sample_results):
    annotated = res['ranked']
    ranks  = list(range(1, len(annotated)+1))
    scores = [s for _, s, _ in annotated]
    is_gold = [g for _, _, g in annotated]

    colors = ['green' if g else 'steelblue' for g in is_gold]
    ax.scatter(ranks, scores, c=colors, s=20, alpha=0.7, zorder=3)
    ax.plot(ranks, scores, color='gray', linewidth=0.5, alpha=0.4)

    # Mark largest gap
    if len(scores) > 1:
        gaps = [scores[j] - scores[j+1] for j in range(len(scores)-1)]
        lg   = int(np.argmax(gaps)) + 1
        ax.axvline(lg + 0.5, color='tomato', linestyle='--', linewidth=1.2, label=f'LG cut={lg}')

    ax.axvline(res['optimal_cut'] + 0.5, color='gold', linestyle=':', linewidth=1.5,
               label=f'optimal={res["optimal_cut"]}')

    ax.set_title(f'n_gold={res["n_gold"]}  opt_F1={res["optimal_f1"]:.2f}', fontsize=9)
    ax.set_xlabel('Rank', fontsize=8)
    ax.set_ylabel('Score', fontsize=8)
    ax.legend(fontsize=7)

gold_patch  = mpatches.Patch(color='green',    label='gold citation')
blue_patch  = mpatches.Patch(color='steelblue',label='non-gold')
fig.legend(handles=[gold_patch, blue_patch], loc='upper right', fontsize=10)
plt.suptitle('Reranker score by rank — green=gold, red dashed=largest gap, yellow=optimal cut',
             fontsize=12)
plt.tight_layout()
plt.show()

## Section 4 — Aggregate gap analysis

Key question: does the **largest gap position** correlate with the **optimal cutoff** across all training queries?

In [ ]:
lg_pos  = np.array([r['largest_gap_pos'] for r in train_results])
opt_cut = np.array([r['optimal_cut']     for r in train_results])
error   = lg_pos - opt_cut   # positive = overshoot, negative = undershoot

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Scatter: largest_gap_pos vs optimal_cut
axes[0].scatter(opt_cut, lg_pos, alpha=0.3, s=15, color='steelblue')
lim = max(opt_cut.max(), lg_pos.max()) + 1
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='perfect agreement')
axes[0].set_xlabel('Optimal cutoff (max-F1)')
axes[0].set_ylabel('Largest gap position')
axes[0].set_title('Largest gap vs optimal cut\n(ideal = on the diagonal)')
axes[0].legend()

# 2. Histogram of error
axes[1].hist(error, bins=range(error.min(), error.max()+2), color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linewidth=1.5)
axes[1].set_xlabel('Error (largest_gap_pos − optimal_cut)\nnegative=undershoot, positive=overshoot')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Gap error distribution\nmean={error.mean():.1f}, median={np.median(error):.1f}')

# 3. Breakdown by citation count bucket
n_golds = np.array([r['n_gold'] for r in train_results])
buckets = [1, 2, 4, 9, 100]
labels  = ['1', '2-3', '4-8', '9+']
bucket_errors = []
for lo, hi in zip(buckets[:-1], buckets[1:]):
    mask = (n_golds >= lo) & (n_golds < hi)
    bucket_errors.append(error[mask])

axes[2].boxplot(bucket_errors, labels=labels, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].axhline(0, color='red', linewidth=1.5)
axes[2].set_xlabel('Gold citation count bucket')
axes[2].set_ylabel('Error (lg_pos − optimal)')
axes[2].set_title('Gap error by citation count bucket')

plt.tight_layout()
plt.show()

print(f'Queries where largest_gap == optimal_cut: {(error==0).sum()}/{len(error)} '
      f'({(error==0).mean():.1%})')
print(f'Queries within ±1:  {(np.abs(error)<=1).mean():.1%}')
print(f'Queries within ±3:  {(np.abs(error)<=3).mean():.1%}')
print(f'Overshoot (lg > opt): {(error>0).mean():.1%}  |  Undershoot: {(error<0).mean():.1%}')

## Section 5 — Strategy comparison by F1

Sweep parameters for each strategy, compare median F1 on train_clean against fixed top-10 baseline.

In [ ]:
strategies = {
    'fixed_k=10':       ('fixed_k',       10),
    'fixed_k=20':       ('fixed_k',       20),
    'largest_gap':      ('largest_gap',   None),
    'gap_thresh=0.05':  ('gap_threshold', 0.05),
    'gap_thresh=0.10':  ('gap_threshold', 0.10),
    'gap_thresh=0.15':  ('gap_threshold', 0.15),
    'relative=0.50':    ('relative',      0.50),
    'relative=0.30':    ('relative',      0.30),
    'relative=0.20':    ('relative',      0.20),
    'oracle':           None,   # optimal per-query
}

strategy_f1s = {}
for name, cfg in strategies.items():
    f1s = []
    for res in train_results:
        gold = set(c for c, _, g in res['ranked'] if g)
        if cfg is None:  # oracle
            f1s.append(res['optimal_f1'])
        else:
            strat, param = cfg
            pred = set(apply_cutoff([(c, s) for c, s, _ in res['ranked']], strat, param))
            f1s.append(f1_score(pred, gold))
    strategy_f1s[name] = f1s

# Summary table
summary = pd.DataFrame({
    name: {
        'median_F1': np.median(f1s),
        'mean_F1':   np.mean(f1s),
        'pct_zero':  np.mean(np.array(f1s) == 0),
    }
    for name, f1s in strategy_f1s.items()
}).T.sort_values('median_F1', ascending=False)
print(summary.to_string(float_format='{:.4f}'.format))

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
names   = list(strategy_f1s.keys())
medians = [np.median(strategy_f1s[n]) for n in names]
colors  = ['gold' if n == 'oracle' else ('tomato' if n == 'fixed_k=10' else 'steelblue') for n in names]
ax.bar(names, medians, color=colors, edgecolor='white')
ax.set_ylabel('Median F1 on train_clean')
ax.set_title('Strategy comparison — median F1')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Section 6 — Apply best strategy to val (English queries, Helsinki translation)

Val uses English queries → translate to German before reranking (same as submission pipeline).

In [ ]:
# Load Helsinki translator for val queries
transl_tok, transl_model = load_translator()

val_results = []
for _, row in val_df.iterrows():
    gold  = set(c.strip() for c in row['gold_citations'].split(';'))
    q_en  = str(row['query'])
    q_de  = translate_to_german(q_en[:TEXT_TRUNCATE], transl_tok, transl_model)

    q_emb    = encode_query(emb_model, q_en[:TEXT_TRUNCATE])  # embed with English query
    top_hits = retrieve_top_k(q_emb, corpus_embs, TOP_K_RETRIEVAL)
    candidates = [(laws_df.iloc[idx]['citation'], laws_df.iloc[idx]['text']) for idx, _ in top_hits]

    ranked    = rerank_with_scores(reranker, q_de, candidates)  # rerank with German query
    annotated = [(c, s, c in gold) for c, s in ranked]

    val_results.append({
        'query_id': row['query_id'],
        'n_gold':   len(gold),
        'ranked':   annotated,
        'gold':     gold,
    })

del transl_model, transl_tok
gc.collect(); torch.cuda.empty_cache()

# ── Evaluate all strategies on val ────────────────────────────────────────────
print(f'{"Strategy":<22}  {"median F1":>10}  {"mean F1":>10}')
print('-' * 46)
for name, cfg in strategies.items():
    f1s = []
    for res in val_results:
        gold = res['gold']
        if cfg is None:
            best = max(
                f1_score(set(c for c, _, _ in res['ranked'][:k]), gold)
                for k in range(1, len(res['ranked'])+1)
            )
            f1s.append(best)
        else:
            strat, param = cfg
            pred = set(apply_cutoff([(c, s) for c, s, _ in res['ranked']], strat, param))
            f1s.append(f1_score(pred, gold))
    print(f'{name:<22}  {np.median(f1s):>10.4f}  {np.mean(f1s):>10.4f}')